In [18]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from typing import TypedDict, List, Dict
import pandas as pd
from operator import add #ajouter à chaque fois un nouveau état
from typing_extensions import Annotated, TypedDict
from qdrant_client.models import VectorParams, Distance
from langchain.tools import tool
from langchain_qdrant import QdrantVectorStore
from langchain_ollama import ChatOllama
from dotenv import  load_dotenv
from qdrant_client.models import (
    Filter,
    FieldCondition,
    GeoRadius,
    GeoPoint
    )
from qdrant_client import QdrantClient
from langchain_community.embeddings import HuggingFaceEmbeddings

#####   1 -Prétraitement et Construction de la source de données.

In [19]:

#loading environment variables, OpenAI key and LangSmith
load_dotenv()


#model creation
model = ChatOllama(
    model="llama3.2", 
    temperature=0
)

In [20]:
df = pd.read_csv(
    "datatourisme-tour.csv", 
    sep=",", 
    encoding="utf-8"
)

In [21]:
#location details are essential; lines without locations details must be removed.
df = df[df["Latitude"].notna()]
df = df[df["Longitude"].notna()]


# Also check for outliers:
# Latitude: 41°N → 51.5°N Longitude: -5° → 10°
# For metropolitan France.
df = df[
(df["Latitude"] > 41) &
(df["Latitude"] < 52)
] 

df = df[
(df["Longitude"] > -5) &
(df["Longitude"] < 11)
] 

# removes identical lines
df = df.drop_duplicates() 
df = df.drop_duplicates(subset=["Nom_du_POI", "Latitude", "Longitude"])


#remove unusable rows; a POI without a name is of no use.
df = df.dropna(
    subset=["Nom_du_POI"]
)

# replace missing values
df["Description"] = df["Description"].fillna("")

In [22]:
#Document creation
def build_document(row) :
    return f"""
Nom_du_POI : {row['Nom_du_POI']}
Categories_de_POI : {row['Categories_de_POI']}
Adresse_postale : {row['Adresse_postale']}
Code_postal_et_commune: {row['Code_postal_et_commune']}
Description: {row['Description']}
""".strip()

df["document"] = df.apply(
    build_document,
    axis=1
)

In [23]:
# Setting up metadata
def build_metadata(row):
    metadata = {
        "Nom_du_POI" : row['Nom_du_POI'], 
        "Categories_de_POI" : row['Categories_de_POI'], 
        "location": {
            "lat": row["Latitude"],
            "lon": row["Longitude"]
        },
        "Code_postal_et_commune" : row['Code_postal_et_commune']
    }
    return metadata

df["metadata"] = df.apply(
    build_metadata,
    axis=1
)

In [24]:
documents = df["document"].tolist()
metadatas = df["metadata"].tolist()

# Generation of documents and metadata for indexing
from langchain_core.documents import Document
docs_metadata = [
    Document(
        page_content=doc,
        metadata=meta
    )
    for doc, meta in zip(df["document"], df["metadata"])
]

##### 2- vectorisation. 

In [26]:

#Initializing the connection to the vector store server
client = QdrantClient(
    url="http://localhost:6333"
)

#
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

client.recreate_collection(
    collection_name="tourisme_tour_france",
    vectors_config=VectorParams(
        size=384,  # IMPORTANT: depends on your embedding template
        distance=Distance.COSINE
    )
)

#creation of a persistent vector_store
v_stores = QdrantVectorStore(
        client=client,
        collection_name="tourisme_tour_france",
        embedding=embeddings
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6687.93it/s]
/var/folders/kw/zr1physd0vq7109nxm533f4h0000gn/T/ipykernel_3626/1516534034.py:9: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


In [27]:

#initialization and creation of chunks for documents and metadata
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200, 
    add_start_index=True
)
all_splits = text_splitter.split_documents(docs_metadata)
print(len(all_splits))

27543


In [28]:
from qdrant_client.models import Distance, VectorParams

collections = client.get_collections().collections
existing = [c.name for c in collections]

if "tourisme_tour_france" not in existing:
    client.recreate_collection    (
        collection_name="tourisme_tour_france",
        vectors_config=VectorParams(
            size=384,
            distance=Distance.COSINE
        )
    )
v_stores.add_documents(all_splits)

['2f258be152394d43863a0014baa8e9dc',
 'f4fb4af6f4ae4ff4a68ec0320de6e110',
 '8b69843f801042e0b5680f6d4807ba0f',
 '4e0ffa50de404f7c800088852f960794',
 'fe43484bc9fd49529335d20f493846fb',
 '6fe3c1a16a3c4533b1bf1e3f0ea9f346',
 'abbd3d02e35a4852920fa475ef448c53',
 '3e96e0d686ed4e13849188436e3e9ed6',
 '62accffb1abd4e8fa50d040dfd1d353f',
 'bdebbe4d77b04764970f1fafcb812572',
 '31b219200a41440f918355198f1b1946',
 '5c2b1992f5354e42b8ad04bdbc16b92c',
 'a9c34a237b354427afd538eec84343f9',
 '302e5aaa53ab4a21a992e229ea481fc5',
 '7a04d375b97f41df91f9134b03c210b4',
 '056b65ce4e93428e8b784adc1750b402',
 'daca2e503e0840b09668113029ffc420',
 '24dd7928322d4b85bdab87d2285b5c68',
 'd4e2fa1311604027aea954bc32ffc845',
 '97b01d95c3bc4d2787c005573d8175bc',
 '82b64f41d18d43059cdcbf74a4a70f41',
 'e5cd0c176e3249bf9a2eea71c5e1e701',
 'd826b49379194a8c9b3b5efa25e34a07',
 '0955e2b7677a40a7aa0510364f1f4b04',
 '11b90d48b3a340c68238a28a36b2224a',
 '683a889d960a404192b75f14a26b6985',
 'e5f3c55477d7475794c9e928b11b049a',
 

##### 3- Développement des outils 

In [29]:
# Tool to search for POI - Tool 1 : Qdrant vector search
@tool
def search_poi(query:str ) : 
    """
    Recherche des points d'intérêt touristiques en France à l'aide d'une recherche vectorielle dans la base Qdrant.
    À utiliser lorsque l'utilisateur demande des monuments, musées, châteaux, plages, parcs ou tout autre lieu touristique.
    """
    docs = v_stores.similarity_search(query, k=5)

    return[
        {
            "Nom_du_POI": d.metadata.get("Nom_du_POI"),
            "Categories_de_POI": d.metadata.get("Categories_de_POI"),
            "Adresse_postale": d.metadata.get("Adresse_postale"), 
            "Code_postal_et_commune": d.metadata.get("Code_postal_et_commune"), 
            "Description": d.metadata.get("Description") #d.page_content
        }
        for d in docs
    ] 

@tool
def search_nearby(latitude: float, longitude: float, radius_km: float = 10, limit: int = 10):
    """
   
   Recherche des points d'intérêt touristiques en France situés à proximité d'une position géographique donnée.
   À utiliser lorsque l'utilisateur demande des lieux proches d'une ville, d'une adresse ou d'une position. 

    """
    points, _ = client.scroll(
        collection_name="tourisme_tour_france",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="location",
                    geo_radius=GeoRadius(
                        center=GeoPoint(
                            lat=latitude,
                            lon=longitude
                        ),
                        radius=radius_km * 1000  # mètres
                    )
                )
            ]
        ),
        limit=limit,
        with_payload=True
    )

    return [
          {
            "Nom_du_POI": p.payload.get("Nom_du_POI"),
            "Categories_de_POI": p.payload.get("Categories_de_POI"),
            "Adresse_postale" : p.payload.get("Adresse_postale"), 
            "Code_postal_et_commune": p.payload.get("Code_postal_et_commune"),
            "Latitude": p.payload.get("Latitude"),
            "Longitude": p.payload.get("Longitude"),
            "Description": p.payload.get("Description")
        }
        for p in points
    ]

@tool
def format_poi (pois) : 
    """formatage des POIs"""
    return "\n".join([
        f"{p['Nom_du_POI']} - {p['Code_postal_et_commune']} ({p['Description']})"
        for p in pois
    ])

tools = [search_poi, search_nearby, format_poi]
#on va utiliser uen fonctions bindtool pour lier le, modèle avec les toolsq qu'il peut utiliser
tools_by_name = {tool.name: tool for tool in tools}
#on passe au modèle une liste des tools qui seront utilisés par ce dernier
model_with_tools = model.bind_tools(tools)

##### 4- Développement de l’architecture du graphe LangGraph – mise en œuvre du RAG Agentic. 

##### 4.1-défnition du State

In [30]:
#State is important. We keep track of all previous  responses
class State(TypedDict):
    query: str
    messages : Annotated[list, add]
    pois : list[Dict]
    answer: str
    steps : int

##### 4.2-Noeud Agent

In [31]:
# At this node of the graph, the agent maintains the conversational context, 
# which is enriched during subsequent calls, with the aim of producing a 
# more precise response.
from langchain_core.messages import HumanMessage, SystemMessage
def agent(state: State):
    if not state.get("messages") : # LLM subsequent call if required
        messages = [
        SystemMessage(content=
            """
            Tu es un assistant touristique spécialisé sur la France Métropolitaine
               
            """
        ),
        HumanMessage(content=state["query"])
    ] 
    else: 
        messages = state["messages"] #first call only

    # LLM request tools the use of tools (response will containt ToolMessage)
    response = model_with_tools.invoke(messages)

    return {
        "messages": state.get("messages", []) + [response], #System, Human, AiMessage, tool_call -> ToolMessage 
        "steps": state.get("steps", 0) + 1
    }

##### 4.3-  Noeud de Tools

In [32]:
from langgraph.prebuilt import ToolNode
# if needed, LLM request to use all existing tools to answser the question
tool_node = ToolNode(tools) # ToolMessage

##### 4.4 -Fonction de routage

In [33]:
#What next ? tool_call or end the program ? 
def should_continue(state):
    if state.get("steps", 0) > 3:
        return "end"
    last_message = state["messages"][-1]
    #print(type(last_message))
    #print(last_message)

    # decide whether there is a need to call a tool
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    # if not end the program and give hand back to LLM
    return "end"

##### 4.5- Noeud de fin

In [34]:
# we are moving to the end
def finalize(state: State):
      # chercher le dernier message du LLM (pas tool)
    for msg in reversed(state["messages"]):
        if hasattr(msg, "content") and msg.content:
            return {"answer": msg.content}

    return {"answer": "Aucune réponse générée"}

#### 5. Orchestration LangGraph

In [35]:
from langgraph.graph import StateGraph, START,END

graph = StateGraph(State)

graph.add_node("agent", agent)
graph.add_node("tools", tool_node)
graph.add_node("final", finalize)

graph.add_edge(START, "agent")
graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "end": "final"
    }
)
graph.add_edge("tools", "agent")
graph.add_edge("final", END)

app = graph.compile()

#### 6. Exécution

In [37]:
result = app.invoke({
    "query": "Peux-tu comparer les attractions touristiques de Nice et Marseille en termes de plages, culture et ambiance ?",
})
#print(result)
print(result["answer"])

Here are some attractions in Nice and Marseille:

1. **Visiter Marseille en 1 jour - Circuit culturel**: A cultural tour that takes you through the city's historic sites, including museums and historical landmarks.
2. **Sur la route du lys et de la rose de Picardie**: A cycling tour that takes you through the picturesque countryside of Picardy, with stops at charming villages and scenic viewpoints.
3. **Balade vieille ville : zoom sur le quartier du Panier**: A walking tour that explores the historic Old Town of Marseille, including its narrow streets, markets, and cultural attractions.
4. **Visiter Marseille en 1 jour - Circuit entre terre et mer**: A tour that combines land and sea activities, taking you to the city's beaches, ports, and scenic coastal routes.
5. **De Martigues à Marseille à vélo**: A cycling tour that takes you from the town of Martigues to Marseille, offering stunning views of the Mediterranean coast.

These are just a few examples of attractions in Nice and Marsei

In [ ]:
#df.info()
#Conserver uniquement ce qui apporte du contexte touristique.
#print(df.isnull().sum()) # nombre de valeur manquante
#df.head()
#print(df['Description'].isnull().sum())
#print(df['Description'].isnull().sum())